# 01 - Converting HL7 v2 `ORU^R01` into a FHIR Message

This notebook converts the HL7 v2.5.1 `ORU^R01` versions of the
[NHS England Pathology FHIR IG examples](https://simplifier.net/guide/pathology-fhir-implementation-guide/Home/Examples/Examples-Index?version=0.5.1)
(in `examples/hl7v2/ORU_R01/`) into **FHIR Messages** - the FHIR equivalent of the HL7 v2 message,
[Laboratory Result (R01)](https://kevinmayfield.github.io/PathologyAndLaboratory/MessageDefinition-laboratory-result.html)
in this guide. `02-fhir-message-to-fhir-document.ipynb` then turns each message into the FHIR Document
the [NHS England Pathology Laboratory Reporting API](https://kevinmayfield.github.io/PathologyAndLaboratory/PathologyReportingAPI.html)
expects.

It follows the rules on the guide's [Mapping HL7 v2 and FHIR](https://kevinmayfield.github.io/PathologyAndLaboratory/mapping.html)
page, which are based on the HL7 [v2-to-FHIR](https://build.fhir.org/ig/HL7/v2-to-fhir/) ConceptMaps.
The approach is the same as the NW GMSA notebook
[04-laboratory-report-fhir-from-hl7v2](https://github.com/nw-gmsa/Testing/blob/main/notebooks/04-laboratory-report-fhir-from-hl7v2.ipynb):
the conversion is **hand-written Python**, so every mapping decision is visible. Only section 2 knows about
HL7 v2 syntax; after that it is "fields to FHIR".

| HL7 v2 | FHIR |
|---|---|
| MSH | MessageHeader, Bundle.identifier / timestamp |
| PID | Patient |
| PV1-3, PV1-8, ORC-12, ORC-21 | PractitionerRole (requester), with **identifier references** to the Practitioner (GMC / GMP code) and Organization (ODS code) |
| MSH-4, OBX-23 | **identifier reference** to the laboratory Organization (ODS code) |
| ORC / OBR | ServiceRequest - one per OBR |
| OBR (several OBX) | Observation - the test group (organizer), `hasMember` the results |
| OBX | Observation - a result |
| OBRs sharing OBR-3 | DiagnosticReport - one per report |
| SPM | Specimen |

Validation (section 8) uses the HL7 FHIR validator (`validator_cli.jar`) against this guide's package, if the
validator is available. It is slow, so it runs once for all the messages.

In [1]:
import json
import os
import re
import subprocess
import uuid
from collections import Counter
from datetime import datetime
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
V2_DIR = ROOT / "examples" / "hl7v2" / "ORU_R01"
OUT_DIR = ROOT / "examples" / "fhir" / "message"
OUT_DIR.mkdir(parents=True, exist_ok=True)

IG = "https://examplesihe.org.uk"
PROFILE = {name: f"{IG}/StructureDefinition/{name}" for name in [
    "BundleMessage", "MessageHeader", "Patient", "Organization", "Practitioner", "PractitionerRole",
    "LaboratoryOrder", "Specimen", "LaboratoryReport", "LaboratoryResult"]}

NHS_NUMBER = "https://fhir.nhs.uk/Id/nhs-number"
ODS = "https://fhir.nhs.uk/Id/ods-organization-code"
SCT = "http://snomed.info/sct"
LOINC = "http://loinc.org"
UCUM = "http://unitsofmeasure.org"
V2_0203 = "http://terminology.hl7.org/CodeSystem/v2-0203"
V2_0074 = "http://terminology.hl7.org/CodeSystem/v2-0074"
V2_0003 = "http://terminology.hl7.org/CodeSystem/v2-0003"
OBS_CATEGORY = "http://terminology.hl7.org/CodeSystem/observation-category"
INTERPRETATION = "http://terminology.hl7.org/CodeSystem/v3-ObservationInterpretation"

sorted(p.name for p in V2_DIR.glob("*.hl7"))

['CRP-Report-Example.hl7',
 'FBC-Report-Example.hl7',
 'GTT-Unstructured-Report-Example.hl7',
 'HBsAg-Report-Example.hl7',
 'HPV-Report-Example.hl7',
 'HbA1c-Report-Example.hl7',
 'LFT-UandE-Report-Example.hl7',
 'Lipids-HbA1c-Report-Example.hl7',
 'Urine-MCS-Unstructured-Report-Example.hl7']

## 1. The source messages

HL7 v2 segments are separated by carriage returns (`\r`). One message - LFT and U&E, two test groups
from one request - is printed in full; it is used to illustrate each step.

In [2]:
def read_message(path):
    text = path.read_bytes().decode("utf-8")
    return [s for s in text.replace("\r\n", "\r").replace("\n", "\r").split("\r") if s]

messages = {p.stem: read_message(p) for p in sorted(V2_DIR.glob("*.hl7"))}
for segment in messages["LFT-UandE-Report-Example"]:
    print(segment[:160] + ("..." if len(segment) > 160 else ""))

MSH|^~\&|LIMS|Y8J7D|GPSYSTEM|B82033|202203081209+0000||ORU^R01^ORU_R01|NHSE-EX-0007|P|2.5.1|||AL|||UNICODE UTF-8
PID|1||RCB4719263^^^RCB^MR~Y8J7D0381527^^^Y8J7D^PI~9999999468^^^NHS^NH||EDITESTPATIENT^ONE^John||19250127|M|||^^^^B6 5RQ|||||||||||||||||||||01
PV1|1|O|^^^^^^^^PICKERING MEDICAL PRACTICE^B82033|||||C1008215^GASKELL^Gale^^^Dr^^^GMC^^^^DN
ORC|RE|REQ-20220307-000046-001^B82033|REP-20220308-008902^Y8J7D||CM||||202203070948+0000|||C1008215^GASKELL^Gale^^^Dr^^^GMC^^^^DN|||||||||PICKERING MEDICAL PRAC...
OBR|1|REQ-20220307-000046-001^B82033|REP-20220308-008902^Y8J7D|1031081000000108^Liver function tests^SCT|||202203071002+0000|||||||||C1008215^GASKELL^Gale^^^Dr^...
OBX|1|NM|997591000000109^Serum total bilirubin level^SCT^1107221000000109^Total bilirubin substance concentration in serum^SCT||6|umol/L^umol/L^UCUM|<21||||F|||...
OBX|2|NM|1000821000000103^Serum albumin level^SCT^1105861000000106^Albumin mass concentration in serum^SCT||47|g/L^g/L^UCUM|35-50||||F|||202203071002+0000||||

## 2. Segments to fields (the only HL7 v2 specific code)

`field(segment, n)` returns field *n* (for MSH, MSH-1 is the field separator, so MSH fields are shifted by
one). Components are split on `^`, sub-components on `&`, repeats on `~`. Escape sequences (`\F\`, `\S\`,
`\T\`, `\R\`, `\E\`, `\.br\`) are undone last, on the final value.

The message is then grouped the way the ORU_R01 abstract message structure groups it: one PID/PV1, then
ORDER_OBSERVATION groups (ORC, OBR, NTE, OBSERVATION groups of OBX + NTE, SPECIMEN groups of SPM).

In [3]:
def fields_of(segment):
    parts = segment.split("|")
    if parts[0] == "MSH":
        parts = ["MSH", "|"] + parts[1:]
    return parts

def field(seg, n):
    return seg[n] if n < len(seg) else ""

def comp(value, n, sep="^"):
    """1-based component (or sub-component with sep="&")."""
    parts = value.split(sep)
    return parts[n - 1] if n - 1 < len(parts) else ""

def unescape(text):
    return (text.replace("\\.br\\", "\n").replace("\\F\\", "|").replace("\\S\\", "^")
            .replace("\\T\\", "&").replace("\\R\\", "~").replace("\\E\\", "\\"))

def parse(segments):
    msg = {"orders": []}
    last_obx = None
    for raw in segments:
        seg = fields_of(raw)
        name = seg[0]
        if name in ("MSH", "PID", "PV1"):
            msg[name] = seg
        elif name == "ORC":
            msg["orders"].append({"ORC": seg, "OBR": None, "NTE": [], "OBX": [], "SPM": []})
            last_obx = None
        elif name == "OBR":
            if not msg["orders"] or msg["orders"][-1]["OBR"] is not None:
                msg["orders"].append({"ORC": None, "OBR": None, "NTE": [], "OBX": [], "SPM": []})
            msg["orders"][-1]["OBR"] = seg
        elif name == "OBX":
            last_obx = {"OBX": seg, "NTE": []}
            msg["orders"][-1]["OBX"].append(last_obx)
        elif name == "NTE":
            (last_obx["NTE"] if last_obx else msg["orders"][-1]["NTE"]).append(seg)
        elif name == "SPM":
            msg["orders"][-1]["SPM"].append(seg)
            last_obx = None
    return msg

parsed = {name: parse(segs) for name, segs in messages.items()}
lft = parsed["LFT-UandE-Report-Example"]
[(field(o["OBR"], 4).split("^")[1], len(o["OBX"]), len(o["SPM"])) for o in lft["orders"]]

[('Liver function tests', 4, 1), ('Urea and electrolytes level', 5, 1)]

## 3. Data types

These follow the [Data Type Level](https://kevinmayfield.github.io/PathologyAndLaboratory/mapping.html#data-type-level)
mappings:

- **TS** to `dateTime` - keep the time zone; with none, assume UK local time (here we add `+00:00`)
- **CWE** to `CodeableConcept` - components 1-3 and 4-6 become two codings, component 9 the text
- **EI** to `Identifier` - the messages send `value^namespace`, the namespace being the ODS code of the
  organisation that assigned the number (the `assigner`). The FHIR `Identifier.system` is **not** in the
  message: a LIMS does not know it at this stage of the pipeline. The converter (usually the integration
  engine) assigns it from an agreed table keyed on *which field* and *which namespace* - `ID_SYSTEMS` below.
  An unknown namespace gets a predictable fallback URI, and is worth flagging as an onboarding gap
- **XCN** / **XON** to Practitioner / Organization

In [4]:
CODING_SYSTEMS = {"SCT": SCT, "LN": LOINC, "UCUM": UCUM}
AUTHORITY = {"GMC": "https://fhir.hl7.org.uk/Id/gmc-number", "GMP": "https://fhir.hl7.org.uk/Id/gmp-number",
             "NMC": "https://fhir.hl7.org.uk/Id/nmc-number", "HCPC": "https://fhir.hl7.org.uk/Id/hcpc-number"}

def to_datetime(value):
    if not value:
        return None
    m = re.match(r"(\d{4})(\d{2})(\d{2})(\d{2})?(\d{2})?(\d{2})?(?:\.\d+)?([+-]\d{4})?$", value)
    y, mo, d, h, mi, s, tz = m.groups()
    if h is None:
        return f"{y}-{mo}-{d}"
    tz = f"{tz[:3]}:{tz[3:]}" if tz else "+00:00"
    return f"{y}-{mo}-{d}T{h}:{mi or '00'}:{s or '00'}{tz}"

def to_concept(value):
    if not value:
        return None
    coding = []
    for start in (1, 4):
        code_, display, system = comp(value, start), comp(value, start + 1), comp(value, start + 2)
        if code_:
            c = {"system": CODING_SYSTEMS.get(system, f"urn:hl7-v2:coding-system:{system}"), "code": unescape(code_)}
            if display:
                c["display"] = unescape(display)
            coding.append(c)
    concept = {"coding": coding} if coding else {}
    if comp(value, 9) or (not coding and comp(value, 2)):
        concept["text"] = unescape(comp(value, 9) or comp(value, 2))
    return concept

# Agreed per interface: (kind of identifier, namespace / ODS code) -> FHIR Identifier.system.
# These are the systems used in the NHS England examples.
ID_SYSTEMS = {
    ("placer-order", "B82033"): "http://B82033-pickeringmedicalpractice.com/labrequest",
    ("placer-specimen", "B82033"): "http://B82033-pickeringmedicalpractice.com/specimen",
    ("report", "Y8J7D"): "http://Y8J7D-pathlab001.com/report",
    ("accession", "Y8J7D"): "http://Y8J7D-pathlab001.com/specimen",
    # patient identifiers: no system is given in the NHS England examples, so the fallback form is used
}

def identifier_system(kind, namespace):
    return ID_SYSTEMS.get((kind, namespace), f"https://examplesihe.org.uk/Id/{kind}/{namespace}")

def to_identifier(ei, type_code=None, kind=None, sep="^"):
    value, namespace = comp(ei, 1, sep), comp(ei, 2, sep)
    if not value:
        return None
    identifier = {}
    if type_code:
        identifier["type"] = {"coding": [{"system": V2_0203, "code": type_code}]}
    if kind:
        identifier["system"] = identifier_system(kind, namespace)
    identifier["value"] = unescape(value)
    if namespace:
        identifier["assigner"] = {"identifier": {"system": ODS, "value": namespace}}
    return identifier

def stable_id(*parts):
    """The FHIR id is derived from business identifiers (mapping page, Rules), so a corrected
    result updates the same resource rather than creating a new one."""
    return str(uuid.uuid5(uuid.NAMESPACE_URL, "|".join(p or "" for p in parts)))

[to_datetime("202203081423+0000"), to_concept(field(lft["orders"][0]["OBR"], 4)),
 to_identifier(field(lft["orders"][0]["OBR"], 2), "PLAC", "placer-order")]

['2022-03-08T14:23:00+00:00',
 {'coding': [{'system': 'http://snomed.info/sct',
    'code': '1031081000000108',
    'display': 'Liver function tests'}]},
 {'type': {'coding': [{'system': 'http://terminology.hl7.org/CodeSystem/v2-0203',
     'code': 'PLAC'}]},
  'system': 'http://B82033-pickeringmedicalpractice.com/labrequest',
  'value': 'REQ-20220307-000046-001',
  'assigner': {'identifier': {'system': 'https://fhir.nhs.uk/Id/ods-organization-code',
    'value': 'B82033'}}}]

## 4. Patient, organisations and people

PID-3 carries three identifiers: the NHS Number, a hospital number (MRN, `MR`, from York and Scarborough `RCB`)
and the laboratory's own patient number (`PI`, from `Y8J7D`) - see the NW GMSA
[MRN-Mapping](https://github.com/nw-gmsa/Testing/blob/main/MRN-Mapping.md). The patients are **NHS England EDI test patients** (999 999 9468 EDITESTPATIENT ONE and 999 999 9549
EDITESTPATIENT NINE), as used by the [NW GMSA IG](https://nw-gmsa.github.io/en/testing.html#nhs-england-edi--test-patients)
and older versions of NHS England PaLM; they also exist in the NHS England INT environment. PID-8 maps through
[v2-0001-to-administrative-gender](https://kevinmayfield.github.io/PathologyAndLaboratory/ConceptMap-v2-0001-to-administrative-gender.html).

**Practitioners and organisations are referenced by identifier, not included.** HL7 v2 only carries their codes
(ODS code in XON-10 / MSH-4, GMC or GMP code in XCN-1) and a name, so the FHIR Message mirrors that: a
`Reference` with `type` and `identifier` (and the name as `display`), and no Practitioner or Organization
resource in the Bundle. The full resources are looked up when the FHIR Document is built (notebook 02).

The requester is a PractitionerRole built from ORC-12 (Ordering Provider) and ORC-21 (Ordering Facility), with
PV1-8 / PV1-3 as the fallback; the NHS England API reads the requesting ODS code from it.

In [5]:
GENDER = {"F": "female", "M": "male", "O": "other", "U": "unknown", "A": "other", "N": "unknown"}

def entry(resource):
    return {"fullUrl": f"urn:uuid:{resource['id']}", "resource": resource}

def ref(resource):
    return {"reference": f"urn:uuid:{resource['id']}"}

def patient_from(pid):
    """PID-3 repeats: NHS Number (CX-4 NHS, CX-5 NH), hospital number (CX-5 MR) and the laboratory's
    own patient number (CX-5 PI). Local identifiers get their system from ID_SYSTEMS, like order numbers."""
    identifiers, nhs_number = [], None
    for cx in field(pid, 3).split("~"):
        value, authority, id_type = comp(cx, 1), comp(cx, 4), comp(cx, 5)
        if authority == "NHS":
            nhs_number = value
            identifiers.append({"type": {"coding": [{"system": V2_0203, "code": "NH"}]},
                                "system": NHS_NUMBER, "value": value})
        elif value:
            kind = {"MR": "mrn", "PI": "patient-internal"}.get(id_type, id_type.lower())
            identifiers.append({"type": {"coding": [{"system": V2_0203, "code": id_type}]},
                                "system": identifier_system(kind, authority), "value": value,
                                "assigner": {"identifier": {"system": ODS, "value": authority}}})
    patient = {"resourceType": "Patient", "id": stable_id(NHS_NUMBER, nhs_number),
               "meta": {"profile": [PROFILE["Patient"]]},
               "identifier": identifiers,
               "name": [{"family": unescape(comp(field(pid, 5), 1)),
                         "given": [unescape(g) for g in (comp(field(pid, 5), 2), comp(field(pid, 5), 3)) if g]}],
               "gender": GENDER.get(field(pid, 8), "unknown"),
               "birthDate": to_datetime(field(pid, 7))}
    if comp(field(pid, 11), 5):
        patient["address"] = [{"postalCode": unescape(comp(field(pid, 11), 5))}]
    return patient, nhs_number

def organization_ref(xon=None, ods_code=None, name=None):
    """XON (or an ODS code) -> identifier reference to the Organization."""
    ods_code = ods_code or comp(xon, 10)
    return {"type": "Organization", "identifier": {"system": ODS, "value": ods_code},
            "display": unescape(name or comp(xon, 1) or ods_code)}

def practitioner_ref(xcn):
    """XCN -> identifier reference to the Practitioner (GMC / GMP / NMC code)."""
    system = AUTHORITY.get(comp(xcn, 9), "https://examplesihe.org.uk/Id/local-practitioner")
    display = " ".join(filter(None, [comp(xcn, 6), unescape(comp(xcn, 3)), unescape(comp(xcn, 2))]))
    return {"type": "Practitioner", "identifier": {"system": system, "value": comp(xcn, 1)}, "display": display}

def practitioner_role_from(practitioner, organization):
    return {"resourceType": "PractitionerRole",
            "id": stable_id("role", practitioner["identifier"]["value"], organization["identifier"]["value"]),
            "meta": {"profile": [PROFILE["PractitionerRole"]]},
            "practitioner": practitioner,
            "organization": organization}

patient, nhs_number = patient_from(lft["PID"])
patient, practitioner_ref(field(lft["orders"][0]["ORC"], 12)), organization_ref(field(lft["orders"][0]["ORC"], 21))

({'resourceType': 'Patient',
  'id': 'bfe333d7-2a8c-5dae-a44a-5d3b120d2ef5',
  'meta': {'profile': ['https://examplesihe.org.uk/StructureDefinition/Patient']},
  'identifier': [{'type': {'coding': [{'system': 'http://terminology.hl7.org/CodeSystem/v2-0203',
       'code': 'MR'}]},
    'system': 'https://examplesihe.org.uk/Id/mrn/RCB',
    'value': 'RCB4719263',
    'assigner': {'identifier': {'system': 'https://fhir.nhs.uk/Id/ods-organization-code',
      'value': 'RCB'}}},
   {'type': {'coding': [{'system': 'http://terminology.hl7.org/CodeSystem/v2-0203',
       'code': 'PI'}]},
    'system': 'https://examplesihe.org.uk/Id/patient-internal/Y8J7D',
    'value': 'Y8J7D0381527',
    'assigner': {'identifier': {'system': 'https://fhir.nhs.uk/Id/ods-organization-code',
      'value': 'Y8J7D'}}},
   {'type': {'coding': [{'system': 'http://terminology.hl7.org/CodeSystem/v2-0203',
       'code': 'NH'}]},
    'system': 'https://fhir.nhs.uk/Id/nhs-number',
    'value': '9999999468'}],
  'name':

## 5. Results - OBX to Observation

[OBX-2 Value Type](https://kevinmayfield.github.io/PathologyAndLaboratory/mapping.html#obx-2-value-type) decides
`value[x]`: `NM` is a Quantity (units from OBX-6), `SN` a Quantity with comparator or a Range, `CWE` a
CodeableConcept, `ST` / `FT` / `TX` a string (with `\.br\` turned back into new lines). OBX-7 is parsed into
`referenceRange` (`low-high`, `<high`, `>low`), OBX-8 and OBX-11 go through the ConceptMaps
[v2-0078](https://kevinmayfield.github.io/PathologyAndLaboratory/ConceptMap-v2-0078-to-observation-interpretation.html) and
[v2-0085](https://kevinmayfield.github.io/PathologyAndLaboratory/ConceptMap-v2-0085-to-observation-status.html), and the NTE
after the OBX become `note`.

In [6]:
OBS_STATUS = {"C": "corrected", "D": "entered-in-error", "F": "final", "I": "registered", "N": "cancelled",
              "O": "registered", "P": "preliminary", "R": "preliminary", "S": "preliminary", "U": "final",
              "W": "entered-in-error", "X": "cancelled"}
INTERPRETATION_DISPLAY = {"L": "Low", "H": "High", "N": "Normal", "LL": "Critical low", "HH": "Critical high",
                          "A": "Abnormal", "AA": "Critical abnormal"}

def quantity(value, units):
    q = {"value": float(value) if "." in value else int(value)}
    if comp(units, 2) or comp(units, 1):
        q["unit"] = unescape(comp(units, 2) or comp(units, 1))
    if comp(units, 3) == "UCUM" and comp(units, 1):
        q["system"], q["code"] = UCUM, unescape(comp(units, 1))
    return q

def set_value(obs, value_type, value, units):
    if not value:
        obs["dataAbsentReason"] = {"coding": [{"system": "http://terminology.hl7.org/CodeSystem/data-absent-reason",
                                               "code": "unknown"}]}
    elif value_type == "NM":
        obs["valueQuantity"] = quantity(value, units)
    elif value_type == "SN":
        comparator, n1, separator, n2 = (comp(value, i) for i in (1, 2, 3, 4))
        if separator == "-":
            obs["valueRange"] = {"low": quantity(n1, units), "high": quantity(n2, units)}
        else:
            obs["valueQuantity"] = {**quantity(n1, units), **({"comparator": unescape(comparator)} if comparator else {})}
    elif value_type in ("CWE", "CE", "CNE"):
        obs["valueCodeableConcept"] = to_concept(value)
    else:  # ST, FT, TX
        obs["valueString"] = unescape(value)

def reference_range(text, units):
    if not text:
        return None
    m = re.match(r"^\s*([\d.]+)\s*-\s*([\d.]+)\s*$", text)
    if m:
        return [{"low": quantity(m.group(1), units), "high": quantity(m.group(2), units)}]
    m = re.match(r"^\s*([<>])=?\s*([\d.]+)\s*$", text)
    if m:
        return [{("high" if m.group(1) == "<" else "low"): quantity(m.group(2), units)}]
    return [{"text": unescape(text)}]

def observation_from(obx_group, common):
    obx = obx_group["OBX"]
    obs = {"resourceType": "Observation",
           "id": stable_id(common["report_key"], field(obx, 3).split("^")[0], field(obx, 4), field(obx, 1),
                           common["obr_set_id"]),
           "meta": {"profile": [PROFILE["LaboratoryResult"]]},
           "basedOn": [ref(common["service_request"])],
           "status": OBS_STATUS.get(field(obx, 11), "final"),
           "category": [{"coding": [{"system": OBS_CATEGORY, "code": "laboratory", "display": "Laboratory"}]}],
           "code": to_concept(field(obx, 3)),
           "subject": common["subject"],
           "effectiveDateTime": to_datetime(field(obx, 14)) or common["effective"],
           "issued": common["issued"],
           "performer": [common["performer"]]}
    set_value(obs, field(obx, 2), field(obx, 5), field(obx, 6))
    if field(obx, 8):
        flag = field(obx, 8)
        obs["interpretation"] = [{"coding": [{"system": INTERPRETATION, "code": flag,
                                              **({"display": INTERPRETATION_DISPLAY[flag]} if flag in INTERPRETATION_DISPLAY else {})}]}]
    if obx_group["NTE"]:
        obs["note"] = [{"text": unescape(field(nte, 3))} for nte in obx_group["NTE"]]
    if common["specimens"]:
        obs["specimen"] = ref(common["specimens"][0])
    rr = reference_range(field(obx, 7), field(obx, 6))
    if rr:
        obs["referenceRange"] = rr
    return obs

## 6. Orders, specimens, test groups and the report

The rules from the mapping page that matter most here:

- **One ServiceRequest per OBR**, identified by ORC-2 / OBR-2 (`PLAC`), with OBR-4 as `code`
- **A test group is the OBR**: when an OBR has several OBX it becomes an Observation with OBR-4 as `code` and
  the OBX as `hasMember` (NHS England calls this an *organizer*); an OBR with one OBX is just that result
- **One DiagnosticReport per report**: the OBRs sharing OBR-3. Its `code` is the report type (SNOMED CT
  721981007 *Diagnostic studies report*, as NHS England, and LOINC 11502-2), its `status` comes from OBR-25
  through [v2-0123](https://kevinmayfield.github.io/PathologyAndLaboratory/ConceptMap-v2-0123-to-diagnostic-report-status.html),
  and the NTE after the OBR become `conclusion`
- **Stable ids**: every `id` is a UUID derived from business identifiers, so re-sending a corrected report gives
  the same resources

In [7]:
REPORT_STATUS = {"O": "registered", "I": "registered", "S": "partial", "A": "partial", "P": "preliminary",
                 "C": "corrected", "R": "partial", "F": "final", "X": "cancelled"}
REPORT_CODE = {"coding": [{"system": SCT, "code": "721981007", "display": "Diagnostic studies report"},
                          {"system": LOINC, "code": "11502-2", "display": "Laboratory report"}]}
SECTION_DISPLAY = {"CH": "Chemistry", "HM": "Hematology", "MB": "Microbiology", "SR": "Serology",
                   "CP": "Cytopathology", "IMM": "Immunology", "VR": "Virology", "LAB": "Laboratory"}

def specimen_from(spm, subject, service_request):
    spm2 = field(spm, 2)
    placer = to_identifier(comp(spm2, 1), "PLAC", "placer-specimen", "&")
    filler = to_identifier(comp(spm2, 2), "ACSN", "accession", "&")
    spec = {"resourceType": "Specimen",
            "id": stable_id((filler or placer).get("system"), (filler or placer)["value"]),
            "meta": {"profile": [PROFILE["Specimen"]]},
            "status": "available",
            "type": to_concept(field(spm, 4)),
            "subject": subject,
            "request": [ref(service_request)],
            "collection": {"collectedDateTime": to_datetime(comp(field(spm, 17), 1))}}
    if placer:
        placer.pop("assigner", None)
        spec["identifier"] = [placer]
    if filler:
        spec["accessionIdentifier"] = filler
    if field(spm, 18):
        spec["receivedTime"] = to_datetime(field(spm, 18))
    return spec

def convert(msg, name):
    msh, pid, pv1 = msg["MSH"], msg["PID"], msg.get("PV1", [])
    resources = {}
    def add(resource):
        resources.setdefault((resource["resourceType"], resource["id"]), resource)
        return resources[(resource["resourceType"], resource["id"])]

    patient, nhs = patient_from(pid)
    add(patient)
    subject = {**ref(patient), "identifier": {"system": NHS_NUMBER, "value": nhs}}
    first_obx = next(g["OBX"] for o in msg["orders"] for g in o["OBX"])
    lab = (organization_ref(field(first_obx, 23)) if field(first_obx, 23)
           else organization_ref(ods_code=comp(field(msh, 4), 1)))

    reports = {}
    for order in msg["orders"]:
        orc, obr = order["ORC"] or [], order["OBR"]
        requester_org_xon = field(orc, 21) or ""
        requester_org = (organization_ref(requester_org_xon) if requester_org_xon else
                         organization_ref(ods_code=comp(field(pv1, 3), 10), name=comp(field(pv1, 3), 9)))
        practitioner = practitioner_ref(field(orc, 12) or field(obr, 16) or field(pv1, 8))
        role = add(practitioner_role_from(practitioner, requester_org))

        placer = to_identifier(field(orc, 2) or field(obr, 2), "PLAC", "placer-order")
        filler = to_identifier(field(obr, 3) or field(orc, 3), "FILL", "report")
        report_key = f"{filler.get('system')}|{filler['value']}"
        sr = {"resourceType": "ServiceRequest", "id": stable_id(placer.get("system"), placer["value"]),
              "meta": {"profile": [PROFILE["LaboratoryOrder"]]},
              "identifier": [placer, filler],
              "status": "completed" if field(orc, 5) == "CM" else "active",
              "intent": "order",
              "code": to_concept(field(obr, 4)),
              "subject": subject,
              "requester": ref(role),
              "performer": [lab]}
        if field(orc, 9):
            sr["authoredOn"] = to_datetime(field(orc, 9))
        sr = add(sr)

        specimens = [add(specimen_from(spm, subject, sr)) for spm in order["SPM"]]
        if specimens:
            sr["specimen"] = [ref(s) for s in specimens]
        common = {"report_key": report_key, "obr_set_id": field(obr, 1), "service_request": sr, "subject": subject,
                  "effective": to_datetime(field(obr, 7)), "issued": to_datetime(field(obr, 22)),
                  "performer": lab, "specimens": specimens}
        results = [add(observation_from(g, common)) for g in order["OBX"]]

        if len(results) > 1:  # the OBR is a test group (organizer)
            group = add({"resourceType": "Observation", "id": stable_id(report_key, "group", field(obr, 1)),
                         "meta": {"profile": [PROFILE["LaboratoryResult"]]},
                         "basedOn": [ref(sr)],
                         "status": OBS_STATUS.get(field(obr, 25), "final"),
                         "category": results[0]["category"],
                         "code": to_concept(field(obr, 4)),
                         "subject": subject, "effectiveDateTime": common["effective"], "issued": common["issued"],
                         "performer": [lab],
                         **({"specimen": ref(specimens[0])} if specimens else {}),
                         "hasMember": [ref(r) for r in results]})
            top = group
        else:
            top = results[0]

        report = reports.get(report_key)
        if report is None:
            report = add({"resourceType": "DiagnosticReport",
                          "id": stable_id(filler.get("system"), filler["value"]),
                          "meta": {"profile": [PROFILE["LaboratoryReport"]]},
                          "identifier": [filler],
                          "basedOn": [], "status": REPORT_STATUS.get(field(obr, 25), "final"),
                          "category": [{"coding": [{"system": V2_0074, "code": field(obr, 24),
                                                    "display": SECTION_DISPLAY.get(field(obr, 24), field(obr, 24))}]}],
                          "code": REPORT_CODE, "subject": subject,
                          "effectiveDateTime": common["effective"], "issued": common["issued"],
                          "performer": [lab], "specimen": [], "result": []})
            reports[report_key] = report
        report["basedOn"].append(ref(sr))
        report["result"].append(ref(top))
        report["specimen"] += [ref(s) for s in specimens if ref(s) not in report["specimen"]]
        if order["NTE"]:
            report["conclusion"] = "\n".join(unescape(field(n, 3)) for n in order["NTE"])

    for report in reports.values():
        if not report["specimen"]:
            del report["specimen"]

    message_id = field(msh, 10)
    header = {"resourceType": "MessageHeader", "id": stable_id("MessageHeader", comp(field(msh, 3), 1), message_id),
              "meta": {"profile": [PROFILE["MessageHeader"]], "lastUpdated": to_datetime(field(msh, 7))},
              "eventCoding": {"system": V2_0003, "code": comp(field(msh, 9), 2),
                              "display": "ORU/ACK - Unsolicited transmission of an observation message"},
              "destination": [{"name": comp(field(msh, 5), 1), "endpoint": f"urn:hl7-v2:{comp(field(msh, 5), 1)}",
                               "receiver": {"identifier": {"system": ODS, "value": comp(field(msh, 6), 1)}}}],
              "sender": {"identifier": {"system": ODS, "value": comp(field(msh, 4), 1)}},
              "source": {"name": comp(field(msh, 3), 1), "endpoint": f"urn:hl7-v2:{comp(field(msh, 3), 1)}"},
              "focus": [ref(r) for r in reports.values()],
              "definition": f"{IG}/MessageDefinition/laboratory-result"}
    # A readable id, so the example's page in the IG is Bundle-Message-<example>.html
    return {"resourceType": "Bundle", "id": f"Message-{name}",
            "meta": {"profile": [PROFILE["BundleMessage"]]},
            "identifier": {"value": message_id},
            "type": "message",
            "timestamp": to_datetime(field(msh, 7)),
            "entry": [entry(header)] + [entry(r) for r in resources.values()]}

bundle = convert(lft, "LFT-UandE-Report-Example")
Counter(e["resource"]["resourceType"] for e in bundle["entry"])

Counter({'Observation': 11,
         'ServiceRequest': 2,
         'MessageHeader': 1,
         'Patient': 1,
         'PractitionerRole': 1,
         'Specimen': 1,
         'DiagnosticReport': 1})

The LFT and U&E report: **one** DiagnosticReport, two ServiceRequests (one per OBR), two test group
Observations and nine result Observations - and no Practitioner or Organization resources.

In [8]:
report = next(e["resource"] for e in bundle["entry"] if e["resource"]["resourceType"] == "DiagnosticReport")
by_id = {e["fullUrl"]: e["resource"] for e in bundle["entry"]}
for r in report["result"]:
    group = by_id[r["reference"]]
    print(group["code"]["coding"][0]["display"])
    for m in group.get("hasMember", []):
        o = by_id[m["reference"]]
        v = o.get("valueQuantity", {})
        print(f"    {o['code']['coding'][0]['display']:<55} {v.get('value', '')} {v.get('unit', '')}")

Liver function tests
    Serum total bilirubin level                             6 umol/L
    Serum albumin level                                     47 g/L
    Serum alkaline phosphatase level                        96 iu/L
    Serum alanine aminotransferase level                    15 iu/L
Urea and electrolytes level
    Serum urea level                                        3.7 mmol/L
    Serum sodium level                                      139 mmol/L
    Serum potassium level                                   4.7 mmol/L
    Serum creatinine level                                  73 umol/L
    eGFR (estimated glomerular filtration rate) using creatinine Chronic Kidney Disease Epidemiology Collaboration equation per 1.73 square metres 100 mL/min/1.73m2


## 7. Convert all the examples

Each FHIR Message is written to `examples/fhir/message/<example>.json`.

In [9]:
rows = []
for name, msg in parsed.items():
    b = convert(msg, name)
    (OUT_DIR / f"{name}.json").write_text(json.dumps(b, indent=2, ensure_ascii=False) + "\n")
    counts = Counter(e["resource"]["resourceType"] for e in b["entry"])
    rows.append((name, counts["DiagnosticReport"], counts["ServiceRequest"], counts["Observation"], counts["Specimen"]))

print(f"{'example':<40} {'DR':>3} {'SR':>3} {'Obs':>4} {'Spec':>4}")
for r in rows:
    print(f"{r[0]:<40} {r[1]:>3} {r[2]:>3} {r[3]:>4} {r[4]:>4}")

example                                   DR  SR  Obs Spec
CRP-Report-Example                         1   1    1    1
FBC-Report-Example                         1   1   14    1
GTT-Unstructured-Report-Example            1   1    1    3
HBsAg-Report-Example                       1   1    1    1
HPV-Report-Example                         1   1    1    1
HbA1c-Report-Example                       1   1    1    1
LFT-UandE-Report-Example                   1   2   11    1
Lipids-HbA1c-Report-Example                1   2    6    1
Urine-MCS-Unstructured-Report-Example      1   1    1    1


## 8. Validate

The HL7 FHIR validator checks the messages against this guide's profiles (the `meta.profile` on each
resource). Set `VALIDATOR_JAR` to the path of `validator_cli.jar`
([download](https://github.com/hapifhir/org.hl7.fhir.core/releases/latest/download/validator_cli.jar));
the guide's package is `output/package.tgz` after a build (`./_genonce.sh`). The validator uses
tx.fhir.org for SNOMED CT (UK Edition) and LOINC.

In [10]:
VALIDATOR = os.getenv("VALIDATOR_JAR", str(ROOT / "validator_cli.jar"))
PACKAGE = ROOT / "output" / "package.tgz"
# Terminology server - tx.fhir.org by default (slow: several minutes per Bundle). Set VALIDATOR_TX=n/a
# to skip terminology checks; the SNOMED CT and LOINC codes are then not checked.
TX = os.getenv("VALIDATOR_TX", "http://tx.fhir.org")
print("terminology server:", TX)

def validate(paths, igs, extra=()):
    out = ROOT / ".cache" / "validation-messages.json"
    out.parent.mkdir(exist_ok=True)
    cmd = ["java", "-jar", VALIDATOR, *map(str, paths), "-version", "4.0.1",
           *[a for ig in igs for a in ("-ig", str(ig))],
           "-tx", TX, *([] if TX == "n/a" else ["-sct", "uk"]), "-output", str(out), *extra]
    subprocess.run(cmd, capture_output=True, text=True)
    return json.loads(out.read_text())

def summarise(result):
    outcomes = result["entry"] if result.get("resourceType") == "Bundle" else [{"resource": result}]
    for e in outcomes:
        oo = e["resource"]
        source = next((x["valueString"] for x in oo.get("extension", []) if x["url"].endswith("operationoutcome-file")), "")
        issues = Counter(i["severity"] for i in oo.get("issue", []))
        print(f"{Path(source).name:<45} errors={issues['error'] + issues['fatal']:<3} warnings={issues['warning']}")
        for i in oo.get("issue", []):
            if i["severity"] in ("error", "fatal"):
                print("    ERROR", i.get("expression", [""])[0], "-", i["details"]["text"][:200])

if Path(VALIDATOR).exists() and PACKAGE.exists():
    summarise(validate(sorted(OUT_DIR.glob("*.json")), [PACKAGE]))
else:
    print("Skipped: set VALIDATOR_JAR and build the IG (./_genonce.sh) to validate")

terminology server: n/a


CRP-Report-Example.json                       errors=0   warnings=7
FBC-Report-Example.json                       errors=0   warnings=20
GTT-Unstructured-Report-Example.json          errors=0   warnings=9
HBsAg-Report-Example.json                     errors=0   warnings=7
HPV-Report-Example.json                       errors=0   warnings=7
HbA1c-Report-Example.json                     errors=0   warnings=8
LFT-UandE-Report-Example.json                 errors=0   warnings=41
Lipids-HbA1c-Report-Example.json              errors=0   warnings=17
Urine-MCS-Unstructured-Report-Example.json    errors=0   warnings=7


## 9. Did anything get lost?

Compare every result in the original NHS England FHIR Documents (downloaded by
`scripts/nhse_document_to_oru.py` into `.cache/`) with the results in the FHIR Messages: same code, same value.
The expected differences are the ones listed on the
[NHS England Examples](https://kevinmayfield.github.io/PathologyAndLaboratory/NHSEnglandExamples.html) page
(placeholder patient details, reference-range text sent as a comment).

In [11]:
import importlib.util
spec = importlib.util.spec_from_file_location("nhse", ROOT / "scripts" / "nhse_document_to_oru.py")
nhse = importlib.util.module_from_spec(spec)
spec.loader.exec_module(nhse)
source_dir = nhse.load_package()

def results(bundle):
    out = set()
    for e in bundle["entry"]:
        o = e["resource"]
        if o["resourceType"] == "Observation" and not o.get("hasMember"):
            value = o.get("valueQuantity", {}).get("value") or o.get("valueString", "").replace("\\n", "\n") \
                or json.dumps(o.get("valueRange") or o.get("valueCodeableConcept"))
            out.add((o["code"]["coding"][0]["code"], str(value)))
    return out

for name, file, *_ in nhse.EXAMPLES:
    original = results(json.loads((source_dir / file).read_text()))
    converted = results(json.loads((OUT_DIR / f"{name}.json").read_text()))
    print(f"{name:<40} results={len(original):>2}  missing={len(original - converted)}  extra={len(converted - original)}")

# Identifiers: the systems assigned from ID_SYSTEMS match the NHS England originals
def identifiers(bundle):
    out = set()
    for e in bundle["entry"]:
        r = e["resource"]
        for i in r.get("identifier", []) if isinstance(r.get("identifier"), list) else []:
            if r["resourceType"] in ("ServiceRequest", "DiagnosticReport", "Specimen"):
                out.add((i.get("system"), i["value"]))
        if r["resourceType"] == "Specimen" and "accessionIdentifier" in r:
            out.add((r["accessionIdentifier"].get("system"), r["accessionIdentifier"]["value"]))
    return out

for name, file, *_ in nhse.EXAMPLES:
    original = identifiers(json.loads((source_dir / file).read_text()))
    converted = identifiers(json.loads((OUT_DIR / f"{name}.json").read_text()))
    print(f"{name:<40} identifiers={len(original):>2}  missing={len(original - converted)}")

HbA1c-Report-Example                     results= 1  missing=0  extra=0
CRP-Report-Example                       results= 1  missing=0  extra=0
HBsAg-Report-Example                     results= 1  missing=0  extra=0
HPV-Report-Example                       results= 1  missing=0  extra=0
FBC-Report-Example                       results=13  missing=0  extra=0
Lipids-HbA1c-Report-Example              results= 5  missing=0  extra=0
LFT-UandE-Report-Example                 results= 9  missing=0  extra=0
GTT-Unstructured-Report-Example          results= 1  missing=0  extra=0
Urine-MCS-Unstructured-Report-Example    results= 1  missing=0  extra=0
HbA1c-Report-Example                     identifiers= 4  missing=0
CRP-Report-Example                       identifiers= 4  missing=0
HBsAg-Report-Example                     identifiers= 4  missing=0
HPV-Report-Example                       identifiers= 4  missing=0
FBC-Report-Example                       identifiers= 4  missing=0
Lipids-HbA1c-Repo